In [1]:
!C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install datasets evaluate scikit-learn
!C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install accelerate


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\shahj\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

print("Step 1: Loading and downsampling the IMDb dataset...")
# Load the official movie reviews dataset
raw_datasets = load_dataset("imdb")

# Downsample to keep it fast and light
train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(1000))
test_dataset = raw_datasets["test"].shuffle(seed=42).select(range(200))

print("Step 2: Tokenizing the textual reviews...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Truncation ensures reviews don't exceed the 512-token limit of BERT
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print("Step 3: Initializing the model configuration...")
# num_labels=2 because sentiment is binary (0: Negative, 1: Positive)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print("Step 4: Setting up evaluation metrics...")
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("Step 5: Configuring hyperparameters & Training Arguments...")
training_args = TrainingArguments(
    output_dir="./results",          # Where checkpoints are saved
    eval_strategy="epoch",           # Evaluate at the end of every epoch
    learning_rate=2e-5,              # Standard fine-tuning learning rate
    per_device_train_batch_size=8,   # Batch size adjusted for consumer memory
    per_device_eval_batch_size=8,
    num_train_epochs=2,              # 2 iterations over the subset data
    weight_decay=0.01,               # Regularization to prevent overfitting
    logging_steps=10                 # Show updates frequently
)

print("Step 6: Initializing the HF Trainer and starting training...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Run the fine-tuning loop!
trainer.train()

print("\nStep 7: Final Evaluation...")
final_metrics = trainer.evaluate()
print(f"Final Test Accuracy: {final_metrics['eval_accuracy']:.2%}")

Step 1: Loading and downsampling the IMDb dataset...


Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\shahj\.cache\huggingface\datasets\imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Tue Jun 23 14:25:29 2026).


Step 2: Tokenizing the textual reviews...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

...

C:\Users\shahj\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Step 7: Final Evaluation...


C:\Users\shahj\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.449412,0.523743,2,0.760000


Final Test Accuracy: 76.00%


In [2]:
from transformers import pipeline

print("Creating test pipeline...")
# Text classification pipeline using the trainer model
custom_pipeline = pipeline(
    "text-classification", 
    model=trainer.model, 
    tokenizer=tokenizer
)

# Custom reviews
my_reviews = [
    "This movie was an absolute masterpiece, the acting was phenomenal!",
    "I fell asleep after 20 minutes. Total waste of time and money.",
    "The visual effects were stunning, but the story was incredibly boring and predictable."
]

print("\n--- INFERENCE TESTS ---")
results = custom_pipeline(my_reviews)

for review, res in zip(my_reviews, results):
    print(f"\nReview: '{review}'")
    # LABEL_0 = Negative, LABEL_1 = Positive
    sentiment = "POSITIVE" if res['label'] == "LABEL_1" else "NEGATIVE"
    print(f"Predicted Sentiment: {sentiment} (Confidence: {res['score']:.2%})")

Creating test pipeline...

--- INFERENCE TESTS ---

Review: 'This movie was an absolute masterpiece, the acting was phenomenal!'
Predicted Sentiment: POSITIVE (Confidence: 88.70%)

Review: 'I fell asleep after 20 minutes. Total waste of time and money.'
Predicted Sentiment: NEGATIVE (Confidence: 89.62%)

Review: 'The visual effects were stunning, but the story was incredibly boring and predictable.'
Predicted Sentiment: NEGATIVE (Confidence: 88.79%)


In [2]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

print("Step 1: Loading and downsampling the IMDb dataset...")
# Load the official movie reviews dataset
raw_datasets = load_dataset("imdb")

# Downsample to keep it fast and light
train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(1000))
test_dataset = raw_datasets["test"].shuffle(seed=42).select(range(200))

print("Step 2: Tokenizing the textual reviews...")
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    # Truncation ensures reviews don't exceed the 512-token limit of BERT
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=64)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print("Step 3: Initializing the model configuration...")
# num_labels=2 because sentiment is binary (0: Negative, 1: Positive)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print("Step 4: Setting up evaluation metrics...")
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("Step 5: Configuring hyperparameters & Training Arguments...")
training_args = TrainingArguments(
    output_dir="./results",          # Where checkpoints are saved
    eval_strategy="epoch",           # Evaluate at the end of every epoch
    learning_rate=2e-5,              # Standard fine-tuning learning rate
    per_device_train_batch_size=8,   # Batch size adjusted for consumer memory
    per_device_eval_batch_size=8,
    num_train_epochs=2,              # 2 iterations over the subset data
    weight_decay=0.01,               # Regularization to prevent overfitting
    logging_steps=10                 # Show updates frequently
)

print("Step 6: Initializing the HF Trainer and starting training...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Run the fine-tuning loop!
trainer.train()

print("\nStep 7: Final Evaluation...")
final_metrics = trainer.evaluate()
print(f"Final Test Accuracy: {final_metrics['eval_accuracy']:.2%}")

from transformers import pipeline

print("Creating test pipeline...")
# Text classification pipeline using the trainer model
custom_pipeline = pipeline(
    "text-classification", 
    model=trainer.model, 
    tokenizer=tokenizer
)

# Custom reviews
my_reviews = [
    "This movie was an absolute masterpiece, the acting was phenomenal!",
    "I fell asleep after 20 minutes. Total waste of time and money.",
    "The visual effects were stunning, but the story was incredibly boring and predictable."
]

print("\n--- INFERENCE TESTS ---")
results = custom_pipeline(my_reviews)

for review, res in zip(my_reviews, results):
    print(f"\nReview: '{review}'")
    # LABEL_0 = Negative, LABEL_1 = Positive
    sentiment = "POSITIVE" if res['label'] == "LABEL_1" else "NEGATIVE"
    print(f"Predicted Sentiment: {sentiment} (Confidence: {res['score']:.2%})")

# --- SAVING MODEL FOR WEEK 4 DEPLOYMENT ---
print("\nStep 8: Saving the model and tokenizer for deployment...")

# Target folder
output_model_dir = "./my_imdb_model"

# Save model and tokenizer in that folder
trainer.save_model(output_model_dir)
tokenizer.save_pretrained(output_model_dir)

print(f"Successfully saved model : {output_model_dir}")

Step 1: Loading and downsampling the IMDb dataset...


Using the latest cached version of the dataset since imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\shahj\.cache\huggingface\datasets\imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Tue Jun 23 17:10:26 2026).


Step 2: Tokenizing the textual reviews...
Step 3: Initializing the model configuration...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

C:\Users\shahj\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.445642,0.536748,2,0.745000


Final Test Accuracy: 74.50%
Creating test pipeline...

--- INFERENCE TESTS ---

Review: 'This movie was an absolute masterpiece, the acting was phenomenal!'
Predicted Sentiment: POSITIVE (Confidence: 84.93%)

Review: 'I fell asleep after 20 minutes. Total waste of time and money.'
Predicted Sentiment: NEGATIVE (Confidence: 91.22%)

Review: 'The visual effects were stunning, but the story was incredibly boring and predictable.'
Predicted Sentiment: NEGATIVE (Confidence: 85.09%)

Step 8: Saving the model and tokenizer for deployment...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully saved model : ./my_imdb_model
